In [3]:
import pandas as pd
from datasets import load_from_disk
from src.data import DATA_DIR_RAW

In [10]:
from ir_datasets_longeval import load

In [11]:
dataset = load("longeval-2023/2022-06/en")

In [19]:
qrels_deep_31["query_id"].apply(lambda x: dataset.query_id_map[x])

0       69c73910-bf4f-4512-bae9-bb4306c9d6c7
1       69c73910-bf4f-4512-bae9-bb4306c9d6c7
2       69c73910-bf4f-4512-bae9-bb4306c9d6c7
3       69c73910-bf4f-4512-bae9-bb4306c9d6c7
4       69c73910-bf4f-4512-bae9-bb4306c9d6c7
                        ...                 
3095    f2f22ecf-812f-4c92-8f87-04f4cc5aed45
3096    f2f22ecf-812f-4c92-8f87-04f4cc5aed45
3097    f2f22ecf-812f-4c92-8f87-04f4cc5aed45
3098    f2f22ecf-812f-4c92-8f87-04f4cc5aed45
3099    f2f22ecf-812f-4c92-8f87-04f4cc5aed45
Name: query_id, Length: 3100, dtype: object

# LongEval
- To use the click model judgments as context to generate the topics they need to be somewhat reliable. How do the human judgments align with the click model judgments? 

In [15]:
BASEPATH = DATA_DIR_RAW / "datasets" / "long_eval_ictir2025"

In [16]:
def ds_to_qrels(ds_path: str):
    """Load Huggingface Datasets dataset and transform it to the TREC qrels format"""
    dataset = load_from_disk(ds_path).to_pandas()
    dataset["Q0"] = "0"
    return dataset[["query_id", "Q0", "doc_id", "relevance"]]

In [17]:
qrels_deep_31 = ds_to_qrels(BASEPATH / "depth_based_31_50_50")
qrels_deep_45 = ds_to_qrels(BASEPATH / "depth_based_45_25_25")

### Duplicate Qrels

In [8]:
print("Duplecates in dataset:", len(qrels_deep_31) -
      len(qrels_deep_31.drop_duplicates(["query_id", "doc_id"])))
print("Duplecates in dataset:", len(qrels_deep_45) -
      len(qrels_deep_45.drop_duplicates(["query_id", "doc_id"])))

Duplecates in dataset: 180
Duplecates in dataset: 58


In [9]:
print("Duplecates in dataset:", len(qrels_deep_31) -
      len(qrels_deep_31.drop_duplicates(["query_id", "doc_id", "relevance"])))
print("Duplecates in dataset:", len(qrels_deep_45) -
      len(qrels_deep_45.drop_duplicates(["query_id", "doc_id", "relevance"])))

Duplecates in dataset: 52
Duplecates in dataset: 13


### How well aligne the human and click model judgments?
Small overlap: 
- 31 Topics: Shallow and deep have 134 overlapping qrels
- 45 Topics: Shallow and deep have 99 overlapping qrels

Insufficient correlation:
- 31 Topics: 0.07
- 45 Topics: 0.06

In [8]:
from ir_datasets_longeval import load
from sklearn.metrics import cohen_kappa_score

In [9]:
dataset = load("longeval-2023/2022-06/en/non-unified")
qrels = pd.DataFrame(dataset.qrels)

In [10]:
qrels["relevance"] = qrels["relevance"].apply(lambda x: 1 if x >= 1 else 0)

In [11]:
df = qrels.merge(qrels_deep_31, how="inner", left_on=[
                 "query_id", "doc_id"], right_on=["query_id", "doc_id"])

In [12]:
print("Shallow and deep have", len(df), "overlapping qrels")

Shallow and deep have 134 overlapping qrels


In [13]:
cohen_kappa_score(df["relevance_x"].astype(str), df["relevance_y"].astype(str))

0.06749285033365093

In [14]:
df = qrels.merge(qrels_deep_45, how="inner", left_on=[
                 "query_id", "doc_id"], right_on=["query_id", "doc_id"])

In [15]:
print("Shallow and deep have", len(df), "overlapping qrels")

Shallow and deep have 99 overlapping qrels


In [16]:
cohen_kappa_score(df["relevance_x"].astype(str), df["relevance_y"].astype(str))

0.0551053484602918

# Overlap

#### 31 Queries

In [31]:
df = qrels.merge(qrels_deep_31, how="left", left_on=["query_id", "doc_id"], right_on=["query_id", "doc_id"])

In [32]:
# click model judgment not judgeed by humans
train_split = df[(df["relevance_y"].isna()) & (df["query_id"].isin(qrels_deep_31["query_id"]))]

In [33]:
# All queries are judged
train_split["query_id"].nunique()

31

In [34]:
# at least 4 judgments per query not judged by humans
train_split.groupby("query_id")["relevance_x"].count().describe()

count    31.000000
mean     11.000000
std       7.056912
min       4.000000
25%       8.000000
50%       9.000000
75%      10.500000
max      35.000000
Name: relevance_x, dtype: float64

In [35]:
train_split[train_split["relevance_x"] == 0].groupby("query_id")["relevance_x"].count().min()

np.int64(2)

In [36]:
train_split[train_split["relevance_x"] >= 1].groupby("query_id")["relevance_x"].count().min()

np.int64(1)

#### 45 Queries

In [25]:
df = qrels.merge(qrels_deep_45, how="left", left_on=["query_id", "doc_id"], right_on=["query_id", "doc_id"])

In [26]:
# click model judgment not judgeed by humans
train_split = df[(df["relevance_y"].isna()) & (df["query_id"].isin(qrels_deep_45["query_id"]))]

In [27]:
# All queries are judged
train_split["query_id"].nunique()

45

In [ ]:
# at least 8 judgments per query not judged by humans
train_split.groupby("query_id")["relevance_x"].count().describe()

count    45.000000
mean     12.311111
std       6.639490
min       8.000000
25%       9.000000
50%      10.000000
75%      12.000000
max      43.000000
Name: relevance_x, dtype: float64

In [29]:
train_split[train_split["relevance_x"] == 0].groupby("query_id")["relevance_x"].count().min()

np.int64(4)

In [30]:
train_split[train_split["relevance_x"] >= 1].groupby("query_id")["relevance_x"].count().min()

np.int64(1)